# Vision-language baseline (Qwen) - sanity checks

Manual prompting examples for the zero-shot Qwen baseline (paper, Section 6 and Appendix B). The quantitative VLM numbers are produced in `benchmarking/`; example images live in `assets/`.


In [ ]:
import torch

from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from transformers import Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from PIL import Image

if not hasattr(torch.compiler, "is_compiling"):
    torch.compiler.is_compiling = lambda: False

In [ ]:
# !pip install --upgrade transformers
# !pip install --upgrade "huggingface-hub>=0.34.0,<1.0"
# !pip install qwen-vl-utils

### Qwen2.5-VL-7B-Instruct

In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct"
)

In [ ]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "assets/image.jpg"},
            {"type": "image", "image": "assets/transformed_image.jpg"},
            {"type": "text", "text": "Identify the similarities between these images."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

In [ ]:
prompt = """You are given two images: Image A (original) and Image B (transformed).  
Your task is to predict the sequence of transformations applied to Image A to obtain Image B, using only the following allowed operations:  
"noop", "grayscale", "rotate_90", "rotate_180", "rotate_270", "color_jitter", "noise_adding", "crop", "horizontal_flip", "vertical_flip".

- The sequence may contain zero, one, or multiple transformations applied in order.  
- If Image A and Image B are identical, return: ["noop"]  
- If Image B can be obtained by applying a sequence of the allowed transformations (in the correct order), return that sequence as a JSON list, e.g.: ["color_jitter", "noise_adding", "rotate_270", "horizontal_flip"]  
- If the transformation from Image A to Image B requires any operation not in the allowed list (e.g., blur, resize, perspective distortion, custom warping, etc.), or if the images are unrelated, return an empty list: []  

Output only the JSON list. Do not add explanations, comments, or extra text."""

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "assets/dog.jpg"},
            {"type": "image", "image": "assets/transformed_dog.jpg"},
            {"type": "text", "text": prompt},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

### Qwen3-VL-4B-Instruct

In [ ]:
# !pip install transformers==4.57.0
# https://github.com/QwenLM/Qwen3-VL/blob/main/qwen-vl-utils/src/qwen_vl_utils/vision_process.py
# https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct/blob/main/config.json

model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct",
)

# Store processor for preprocessing
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct", )

# vision_model = model.visual

In [ ]:
model.to('cuda')

In [ ]:
# наша красотка занимает 17362MiB / 24576MiB

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "imgs/car/image1.jpg"},
            {"type": "image", "image": "imgs/car/image2.jpg"},
            {"type": "text", "text": "Identify the similarities between these images."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

In [ ]:
image_path = 'assets/image.jpg'

image = Image.open(image_path)
out = processor.image_processor(image)

print(out['pixel_values'].shape, out['image_grid_thw'])